# Notebook 11 — Information-Theoretic Metacognition (`metasignal.itmc`)

This notebook walks through the full `itmc` submodule, which implements the five
information-theoretic (IT) metacognition measures introduced by:

> Dayan P (2023). *Metacognitive Information Theory*. Open Mind, 7, 392–411.
> https://doi.org/10.1162/opmi_a_00091

and ported from the statConfR R package:

> Rausch et al. (2025). statConfR. *JOSS*. https://doi.org/10.21105/joss.06966

## Measures

| Measure | Formula | Meaning |
|---|---|---|
| `meta_I` | H₂(acc) − H₂(acc\|conf) | Mutual information between confidence and accuracy (bits) |
| `meta_Ir1` | meta-I / meta-I_Gaussian(d′) | Efficiency relative to ideal Gaussian observer with same d′ |
| `meta_Ir1_acc` | meta-I / meta-I_Gaussian(acc) | Same but normalised by observed accuracy instead of d′ |
| `meta_Ir2` | meta-I / H₂(acc) | Fraction of maximum possible information, range [0, 1] |
| `RMI` | (meta-I − I_min) / (I_max − I_min) | Range-normalised; 0 = worst, 1 = best metacognition |

## Two backends

| `backend=` | Formula | Best for |
|---|---|---|
| `'simple'` | MI(accuracy; confidence rating) = H₂(acc) − H₂(acc\|conf) | Fast exploration; meta_I, meta_Ir1, meta_Ir2 |
| `'statconfr'` | I(stimulus; graded response) − I_min, with analytic bounds | Exact reproduction of statConfR; RMI |

**Use `backend='statconfr'` for RMI** — the simple-backend bounds are not tight for that measure.

---

In [ ]:
import sys, os, warnings
warnings.filterwarnings('ignore')

REPO = os.path.abspath(os.path.join(
    os.getcwd(), '..' if os.path.basename(os.getcwd()) == 'notebooks' else '.'))
sys.path.insert(0, os.path.join(REPO, 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

from metasignal.itmc import (
    meta_I, meta_Ir1, meta_Ir1_acc, meta_Ir2, RMI,
    estimate_meta_I, fit_group, MEASURE_COLS,
)
from metasignal.stdpy.simulate import trialSimulation, pairedResponseSimulation

plt.style.use('seaborn-v0_8-whitegrid')
COLORS = {'simple': '#4C72B0', 'statconfr': '#DD8452'}
print('Ready.')

## 1. Single-participant quick-start

We simulate one participant using `trialSimulation` (which generates realistic
overlapping confidence distributions from a Gaussian SDT model) and compute all
five measures with both backends.

In [ ]:
rng = np.random.default_rng(0)
df_one = trialSimulation(d=1.5, metad=1.2, nTrials=600, rng=rng)

stim = df_one['Stimuli'].to_numpy(dtype=int)
resp = df_one['Responses'].to_numpy(dtype=int)
conf = df_one['Confidence'].to_numpy(dtype=int)

print(f"Trials: {len(stim)}   Accuracy: {(stim==resp).mean():.2f}")
print(f"Confidence levels: {sorted(set(conf))}")
print()

rows = []
for backend in ['simple', 'statconfr']:
    rows.append({
        'backend':       backend,
        'meta_I':        meta_I(stim, resp, conf, backend=backend),
        'meta_Ir1':      meta_Ir1(stim, resp, conf, backend=backend),
        'meta_Ir1_acc':  meta_Ir1_acc(stim, resp, conf, backend=backend),
        'meta_Ir2':      meta_Ir2(stim, resp, conf, backend=backend),
        'RMI':           RMI(stim, resp, conf, backend=backend),
    })

pd.DataFrame(rows).set_index('backend').round(4)

## 2. Understanding each measure

### 2a. meta-I — raw mutual information

`meta_I` measures how many bits of information confidence ratings carry about
whether the participant was correct.  A value of 0 means confidence is unrelated
to accuracy; higher = better metacognitive sensitivity.

In [ ]:
# Compare informative vs random confidence
rng2 = np.random.default_rng(1)
conf_random = rng2.integers(1, 5, size=len(stim))  # shuffle breaks accuracy-confidence link

mi_real   = meta_I(stim, resp, conf,        backend='statconfr')
mi_random = meta_I(stim, resp, conf_random, backend='statconfr')

print(f"meta-I (informative confidence): {mi_real:.4f} bits")
print(f"meta-I (random confidence):      {mi_random:.4f} bits  ← should be ≈ 0")

### 2b. meta-I₁ʳ — efficiency relative to an ideal Gaussian observer

Values < 1 indicate sub-ideal metacognition (the participant transmits less
metacognitive information than a Gaussian observer with the same d′ would).
Values > 1 are theoretically possible when the participant has access to
additional information sources.

In [ ]:
# Vary meta-d' while holding d' fixed, show how meta_Ir1 tracks it
dprime_fixed = 1.5
metad_levels = [0.5, 0.8, 1.0, 1.2, 1.5]

ir1_vals = []
for md in metad_levels:
    df_tmp = trialSimulation(d=dprime_fixed, metad=md, nTrials=2000,
                             rng=np.random.default_rng(99))
    s = df_tmp['Stimuli'].to_numpy(int)
    r = df_tmp['Responses'].to_numpy(int)
    c = df_tmp['Confidence'].to_numpy(int)
    ir1_vals.append(meta_Ir1(s, r, c, backend='statconfr'))

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(metad_levels, ir1_vals, 'o-', color=COLORS['statconfr'])
ax.axhline(1.0, ls='--', color='gray', lw=0.8, label='Ideal observer')
ax.set_xlabel("meta-d′")
ax.set_ylabel("meta-I₁ʳ")
ax.set_title(f"meta-I₁ʳ vs meta-d′  (d′ fixed at {dprime_fixed})")
ax.legend()
plt.tight_layout()
plt.show()

### 2c. meta-I₂ʳ and RMI

`meta_Ir2` = meta-I / H₂(accuracy): the fraction of the *maximum possible*
information that is actually transmitted.  Range [0, 1].

`RMI` rescales between the analytic minimum and maximum I(S;R) achievable at the
observed accuracy level (Dayan 2023, Theorem 3).  Also [0, 1]; requires
`backend='statconfr'`.

In [ ]:
print(f"meta-I₂ʳ (simple):    {meta_Ir2(stim, resp, conf, backend='simple'):.4f}")
print(f"meta-I₂ʳ (statconfr): {meta_Ir2(stim, resp, conf, backend='statconfr'):.4f}")
print(f"RMI      (statconfr): {RMI(stim, resp, conf, backend='statconfr'):.4f}  ← use statconfr for RMI")

## 3. Bias correction

With small samples, meta-I has a positive sampling bias (MI is always ≥ 0 even
under the null).  `bias_correction=True` subtracts an estimated bias:

- `backend='simple'`: permutes accuracy labels (2000 shuffles)
- `backend='statconfr'`: multinomial resampling of the contingency table (1000 sims),
  matching statConfR's `bias_reduction=TRUE`

In [ ]:
# Small-sample bias demo
rng3 = np.random.default_rng(7)
n_trials_list = [50, 100, 200, 400, 800]

raw_vals, bc_vals = [], []
for n in n_trials_list:
    df_tmp = trialSimulation(d=1.0, metad=0.3, nTrials=n, rng=np.random.default_rng(7))
    s = df_tmp['Stimuli'].to_numpy(int)
    r = df_tmp['Responses'].to_numpy(int)
    c = df_tmp['Confidence'].to_numpy(int)
    raw_vals.append(meta_I(s, r, c, backend='simple', bias_correction=False))
    bc_vals.append( meta_I(s, r, c, backend='simple', bias_correction=True, seed=0))

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(n_trials_list, raw_vals, 'o-', label='Raw meta-I',       color=COLORS['simple'])
ax.plot(n_trials_list, bc_vals,  's--', label='Bias-corrected',  color='#55A868')
ax.set_xlabel('N trials')
ax.set_ylabel('meta-I (bits)')
ax.set_title('Bias correction effect at small N')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Group-level analysis with `estimate_meta_I`

`estimate_meta_I(df)` mirrors statConfR's `estimateMetaI()`: one row per
participant, all five measures.

In [ ]:
# Simulate 10 participants
records = []
for pid in range(10):
    df_p = trialSimulation(d=1.2 + 0.1*pid, metad=0.8 + 0.05*pid, nTrials=400,
                           rng=np.random.default_rng(pid))
    df_p['participant'] = f'p{pid:02d}'
    df_p = df_p.rename(columns={'Stimuli':'stimulus','Responses':'response','Confidence':'rating'})
    records.append(df_p[['stimulus','response','rating','participant']])

df_group = pd.concat(records, ignore_index=True)

result_sc = estimate_meta_I(df_group, backend='statconfr')
result_sc.round(4)

In [ ]:
# Forest plot of meta-I across participants
fig, ax = plt.subplots(figsize=(5, 4))
ax.barh(result_sc['participant'], result_sc['meta_I'], color=COLORS['statconfr'], height=0.6)
ax.set_xlabel('meta-I (bits)')
ax.set_title('meta-I by participant (statconfr backend)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 5. Factorial designs with `fit_group`

`fit_group` supports any combination of `subject`, `within`, and `between`
columns — including lists of multiple factors.  It returns a tidy DataFrame
with one row per cell.

In [ ]:
# 2 conditions (easy / hard) × 8 subjects
records2 = []
for pid in range(8):
    for cond, label in [(0, 'easy'), (1, 'hard')]:
        d    = 2.0 if cond == 0 else 1.0
        df_p = trialSimulation(d=d, metad=d*0.8, nTrials=300,
                               rng=np.random.default_rng(pid*10+cond))
        df_p = df_p.rename(columns={'Stimuli':'stimulus','Responses':'response','Confidence':'rating'})
        df_p['subject']   = f'p{pid:02d}'
        df_p['condition'] = label
        df_p['group']     = 'control' if pid < 4 else 'patient'
        records2.append(df_p[['stimulus','response','rating','subject','condition','group']])

df2 = pd.concat(records2, ignore_index=True)

# Within-subjects (condition), between-subjects (group)
out = fit_group(df2,
                stimuli='stimulus', responses='response', confidence='rating',
                subject='subject', within='condition', between='group',
                backend='statconfr')
print(f"Shape: {out.shape}  — one row per subject × condition")
out.round(4)

In [ ]:
# Visualise: easy vs hard meta-I, split by group
fig, axes = plt.subplots(1, 2, figsize=(8, 3.5), sharey=True)
for ax, grp in zip(axes, ['control', 'patient']):
    sub = out[out['group'] == grp]
    easy = sub[sub['condition'] == 'easy']['meta_I'].values
    hard = sub[sub['condition'] == 'hard']['meta_I'].values
    ax.scatter(easy, hard, color=COLORS['statconfr'], zorder=3)
    lim = max(easy.max(), hard.max()) * 1.1
    ax.plot([0, lim], [0, lim], 'k--', lw=0.8, alpha=0.4)
    ax.set_xlabel('meta-I easy')
    ax.set_ylabel('meta-I hard')
    ax.set_title(grp)
plt.suptitle('meta-I: easy vs hard condition', y=1.02)
plt.tight_layout()
plt.show()

## 6. Multiple within/between factors

Pass a list to `within` or `between` to group by multiple factors simultaneously.

In [ ]:
# Add a second within factor: 'block'
records3 = []
for pid in range(4):
    for cond in [0, 1]:
        for block in [1, 2]:
            df_p = trialSimulation(d=1.2+0.2*pid, metad=1.0, nTrials=200,
                                   rng=np.random.default_rng(pid*100+cond*10+block))
            df_p = df_p.rename(columns={'Stimuli':'stimulus','Responses':'response','Confidence':'rating'})
            df_p['subject']   = f'p{pid}'
            df_p['condition'] = cond
            df_p['block']     = block
            records3.append(df_p[['stimulus','response','rating','subject','condition','block']])

df3 = pd.concat(records3, ignore_index=True)

out3 = fit_group(df3,
                 stimuli='stimulus', responses='response', confidence='rating',
                 subject='subject', within=['condition', 'block'],
                 backend='statconfr')
print(f"Shape: {out3.shape}  — {df3.subject.nunique()} subjects × 2 conditions × 2 blocks")
out3.head(8).round(4)

## 7. Measure subset

Use the `measures` argument to request only specific columns.

In [ ]:
print(f"All available measures: {MEASURE_COLS}")

out_subset = fit_group(df2,
                       stimuli='stimulus', responses='response', confidence='rating',
                       subject='subject', within='condition',
                       measures=['meta_I', 'RMI'],
                       backend='statconfr')
out_subset.head(4).round(4)

## 8. Backend comparison

Both backends return closely-matched values for meta-I, meta-Ir1 and meta-Ir2.
The main differences are in RMI (analytic bounds vs. simplified approximation)
and in the Gaussian reference for meta-Ir1 (numerical integration vs. Monte Carlo).

In [ ]:
both = pd.merge(
    estimate_meta_I(df_group, backend='simple').add_suffix('_simple').rename(columns={'participant_simple':'participant'}),
    estimate_meta_I(df_group, backend='statconfr').add_suffix('_sc').rename(columns={'participant_sc':'participant'}),
    on='participant'
)

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for ax, m in zip(axes, ['meta_I', 'meta_Ir2', 'RMI']):
    x = both[f'{m}_simple']
    y = both[f'{m}_sc']
    ax.scatter(x, y, color='#555', zorder=3)
    mn, mx = min(x.min(), y.min()), max(x.max(), y.max())
    ax.plot([mn, mx], [mn, mx], 'r--', lw=0.8)
    ax.set_xlabel('simple')
    ax.set_ylabel('statconfr')
    ax.set_title(m)
plt.suptitle('Backend comparison across 10 participants', y=1.02)
plt.tight_layout()
plt.show()

# Correlation between backends
for m in ['meta_I', 'meta_Ir1', 'meta_Ir2']:
    r = both[f'{m}_simple'].corr(both[f'{m}_sc'])
    print(f"  {m}: r = {r:.4f}")

## 9. IT measures vs traditional meta-d′ measures

IT measures are model-free: they do not assume a Gaussian signal-detection model.
Here we show that `meta_I` correlates with M-ratio across participants but captures
different variance.

In [ ]:
from metasignal.stdpy import fit_group as stdpy_fit_group

df_group_std = df_group.rename(columns={'stimulus':'Stimuli','response':'Responses','rating':'Confidence'})

trad = stdpy_fit_group(df_group_std, subject='participant', nRatings=4,
                       measures=['dprime','meta_d','M_ratio'])
trad = trad.rename(columns={'participant':'participant'})

it = estimate_meta_I(df_group, backend='statconfr')

merged = pd.merge(trad, it, on='participant')

fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
axes[0].scatter(merged['M_ratio'], merged['meta_I'], color=COLORS['statconfr'])
axes[0].set_xlabel("M-ratio (meta-d′ / d′)")
axes[0].set_ylabel("meta-I (bits)")
axes[0].set_title("meta-I vs M-ratio")

axes[1].scatter(merged['M_ratio'], merged['RMI'], color=COLORS['statconfr'])
axes[1].set_xlabel("M-ratio")
axes[1].set_ylabel("RMI")
axes[1].set_title("RMI vs M-ratio")

plt.tight_layout()
plt.show()

r_mi  = merged['M_ratio'].corr(merged['meta_I'])
r_rmi = merged['M_ratio'].corr(merged['RMI'])
print(f"Pearson r(M-ratio, meta-I) = {r_mi:.3f}")
print(f"Pearson r(M-ratio, RMI)    = {r_rmi:.3f}")

## 10. Summary

| Task | Function | Key argument |
|---|---|---|
| Single measure, one participant | `meta_I(stim, resp, conf)` | `backend=` |
| All five measures, one participant | `estimate_meta_I(df)` | `backend=`, `bias_correction=` |
| All five measures, factorial design | `fit_group(df, subject=, within=, between=)` | `backend=`, `measures=` |

### Backend guide
- **Exploration / speed** → `backend='simple'` (default)
- **Reproducing statConfR results** → `backend='statconfr'`
- **RMI always** → `backend='statconfr'` (simple backend bounds are approximate)
- **Small N / publication** → add `bias_correction=True`

### References
- Dayan P (2023). Metacognitive Information Theory. *Open Mind*, 7, 392–411.
- Rausch M et al. (2025). statConfR. *JOSS*. doi:10.21105/joss.06966